In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="03-google-street-view",
    notebook_name="04_interview_walkthrough.ipynb"
)

# Mock Interview Walkthrough -- Google Street View Blurring System

## What This Notebook Is

This is a **complete mock interview simulation** for the Google Street View blurring system design question. It is structured exactly like a real 45-minute ML system design interview, with:

- Interviewer/candidate dialogue format
- Timing for each section
- Whiteboard-style diagrams
- Common follow-up questions with model answers
- A scoring rubric so you know what "good" looks like
- A 30-second pitch for the elevator ride after the interview

### How to Use This Notebook

1. **First pass**: Read through the dialogue to see what a great answer looks like
2. **Practice**: Cover the candidate answers and try answering yourself
3. **Time yourself**: Can you hit each section within the time budget?
4. **Use the rubric**: Score yourself honestly

---

## The 45-Minute Interview Timeline

| Phase | Time | What Happens |
|-------|------|--------------|
| Problem Statement | 0:00-1:00 | Interviewer sets the scene |
| Clarifying Questions | 1:00-5:00 | YOU ask smart questions |
| ML Framing | 5:00-10:00 | Frame as detection + choose architecture |
| Data Pipeline | 10:00-15:00 | Data sources + augmentation |
| Model Architecture | 15:00-25:00 | Faster R-CNN deep dive + loss function |
| Evaluation | 25:00-32:00 | IoU -> AP -> mAP + online metrics |
| Serving | 32:00-40:00 | NMS + batch pipeline + feedback loop |
| Follow-Up Questions | 40:00-45:00 | Curveballs and deep dives |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch

# === Interview Timeline Visualization ===

phases = [
    ('Clarify Requirements', 0, 5, '#FFE0B2'),
    ('ML Framing', 5, 10, '#B3E5FC'),
    ('Data Pipeline', 10, 15, '#C8E6C9'),
    ('Model Architecture', 15, 25, '#F8BBD0'),
    ('Evaluation', 25, 32, '#D1C4E9'),
    ('Serving', 32, 40, '#FFF9C4'),
    ('Follow-ups', 40, 45, '#FFCDD2'),
]

fig, ax = plt.subplots(figsize=(16, 3))

for name, start, end, color in phases:
    duration = end - start
    rect = FancyBboxPatch((start, 0.2), duration, 0.6, boxstyle='round,pad=0.05',
                          facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(start + duration/2, 0.5, f'{name}\n({duration} min)',
            ha='center', va='center', fontsize=8, fontweight='bold')

ax.set_xlim(-1, 46)
ax.set_ylim(0, 1)
ax.set_xlabel('Minutes', fontsize=12)
ax.set_title('45-Minute ML System Design Interview Timeline', fontsize=14, fontweight='bold')
ax.set_yticks([])

# Add minute markers
for m in range(0, 46, 5):
    ax.axvline(m, color='gray', linewidth=0.5, alpha=0.3)
    ax.text(m, 0.05, f'{m}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---

## Phase 1: Problem Statement (0:00 - 1:00)

---

> **INTERVIEWER**: *"Google Street View has cars that photograph streets around the world. These images are displayed on Google Maps. Design a system that automatically detects and blurs personally identifiable information -- specifically human faces and license plates -- before the images are shown to users."*

---

**What you should be thinking**: "This is an object detection problem. It is offline (batch processing). Privacy means recall is more important than precision. Let me confirm my assumptions."

## Phase 2: Clarifying Questions (1:00 - 5:00)

**Staff-level move**: NEVER jump straight to a solution. Ask 5-7 targeted questions that show you understand the problem space. Each question should reveal a design constraint.

---

> **CANDIDATE**: *"Before I dive in, I would like to clarify a few things. Is it fair to say the business objective is to protect user privacy?"*
>
> **INTERVIEWER**: *"Yes."*
>
> **CANDIDATE**: *"Great. So we want to detect all human faces and license plates in Street View images and blur them before displaying to users. And I assume users can report images that are incorrectly blurred -- either missed faces or things blurred that should not be?"*
>
> **INTERVIEWER**: *"Yes, those are fair assumptions."*
>
> **CANDIDATE**: *"Do we have an annotated dataset?"*
>
> **INTERVIEWER**: *"Yes, 1 million images with bounding box annotations for faces and license plates."*
>
> **CANDIDATE**: *"The dataset might not represent all demographics equally -- race, age, gender -- which could cause bias in face detection. Should we address that today?"*
>
> **INTERVIEWER**: *"Great point, but let's keep it out of scope for this discussion."*
>
> **CANDIDATE**: *"My understanding is that latency is NOT a concern here, since we can display existing images while new ones are processed offline. Is that correct?"*
>
> **INTERVIEWER**: *"Yes. Offline batch processing."*

---

**Why this is a staff-level answer**: Each question reveals a KEY constraint:
1. Privacy = recall matters more than precision
2. User reports = feedback loop exists
3. 1M annotated images = supervised learning is feasible
4. Bias awareness = shows maturity
5. Offline = we can use slow but accurate models

In [ ]:
# === Requirements Summary (the kind of structured thinking interviewers love) ===

requirements = {
    "Business Objective": "Protect user privacy by blurring PII",
    "PII Types": ["Human faces", "License plates"],
    "Processing Mode": "OFFLINE batch processing (no latency constraint)",
    "Training Data": "1M images with bounding box annotations",
    "Feedback": "Users can report missed/incorrect blurs",
    "Key Insight": "Recall >> Precision (missed face = privacy violation, false blur = cosmetic)",
}

print("=" * 60)
print("  REQUIREMENTS SUMMARY")
print("=" * 60)
for key, value in requirements.items():
    if isinstance(value, list):
        print(f"\n  {key}:")
        for item in value:
            print(f"    - {item}")
    else:
        print(f"\n  {key}: {value}")

print("\n" + "=" * 60)
print("\nCRITICAL TRADE-OFF TO STATE UPFRONT:")
print("  False Negative (missed face) = PRIVACY VIOLATION = very bad")
print("  False Positive (blurred sign) = cosmetic issue  = acceptable")
print("  Therefore: OPTIMIZE FOR RECALL, tolerate false positives.")

## Phase 3: ML Framing (5:00 - 10:00)

---

> **CANDIDATE**: *"Let me frame this as an ML problem. The business objective is privacy protection, but the ML objective is to accurately detect objects of interest -- faces and license plates -- in images. If we can detect them reliably, we can blur them before display.*
>
> *"The input is a street-level image. The output is a list of bounding boxes, each with coordinates (x, y, width, height), a class label (face or license plate), and a confidence score.*
>
> *"This is an object detection problem, which combines localization -- a regression task to predict bounding box coordinates -- with classification -- predicting the class of each detected region.*
>
> *"For the architecture, I would choose a two-stage detector, specifically Faster R-CNN. Here is why:*
>
> *1. Privacy is the top priority, so we need maximum accuracy and recall*
> *2. Processing is offline, so slower inference is perfectly acceptable*
> *3. 1M images is manageable -- no need to sacrifice accuracy for training speed*
> *4. If we later need to scale or go real-time, we can switch to YOLO/SSD"*

---

**Why this is strong**: States the ML objective clearly, specifies exact input/output, identifies the ML category, chooses architecture with explicit rationale, and mentions an alternative.

In [ ]:
# === Architecture Comparison (Whiteboard Material) ===

print("=== Architecture Decision (What You'd Draw on Whiteboard) ===")
print()
print("OPTION A: Two-Stage (Faster R-CNN)")
print("  [Image] -> [Backbone CNN] -> [RPN] -> [RoI Pooling] -> [Detection Head] -> [Bboxes]")
print("  Speed: ~5 FPS | mAP: ~42 (COCO) | Best for: accuracy-critical, offline")
print()
print("OPTION B: One-Stage (YOLO)")
print("  [Image] -> [CNN] -> [Grid Prediction] -> [Bboxes]")
print("  Speed: ~60 FPS | mAP: ~37-40 (COCO) | Best for: real-time")
print()
print("OPTION C: Transformer (DETR)")
print("  [Image] -> [CNN + Transformer Encoder] -> [Transformer Decoder] -> [Bboxes]")
print("  Speed: ~15 FPS | mAP: ~42+ (COCO) | Best for: no hand-designed components")
print()
print("=" * 65)
print("OUR CHOICE: Faster R-CNN")
print("  Reason: Offline processing + max accuracy needed for privacy")
print("  Fallback: Switch to YOLO if we need to process 100x more images")
print("=" * 65)

## Phase 4: Data Pipeline (10:00 - 15:00)

---

> **CANDIDATE**: *"For data, we have two sources. First, the annotated dataset: 1 million images with bounding boxes in the format [x, y, width, height] plus class labels. Second, raw Street View images that come with metadata like GPS coordinates, camera pitch/yaw/roll, and timestamps.*
>
> *"For feature engineering, I would apply standard image preprocessing: resize to a consistent input size like 800x800, normalize pixel values to zero mean and unit variance.*
>
> *"For data augmentation, I would use offline augmentation to grow the dataset from 1 million to roughly 10 million images. The key augmentations are horizontal flip, random crop, brightness/contrast changes, and rotation. But critically, for object detection, all geometric transforms must also be applied to the bounding box annotations. If you flip the image, you must flip the box coordinates too -- this is a common gotcha.*
>
> *"I would choose offline augmentation over online because storage is cheap and it avoids augmentation overhead during training, leading to faster iteration."*

---

In [ ]:
# === Data Pipeline Visualization ===

fig, ax = plt.subplots(figsize=(16, 3))
ax.set_xlim(0, 16)
ax.set_ylim(0, 2)
ax.axis('off')
ax.set_title('Data Pipeline (Whiteboard Diagram)', fontsize=14, fontweight='bold')

def draw_pipeline_box(x, y, w, h, text, color):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=9, fontweight='bold')

draw_pipeline_box(0.2, 0.5, 2.5, 1.0, 'Raw Images\n(1M)', '#FFE0B2')
draw_pipeline_box(3.5, 0.5, 2.5, 1.0, 'Preprocessing\n(resize, normalize)', '#B3E5FC')
draw_pipeline_box(6.8, 0.5, 2.5, 1.0, 'Augmentation\n(offline)', '#C8E6C9')
draw_pipeline_box(10.1, 0.5, 2.5, 1.0, 'Train/Val/Test\nSplit (80/10/10)', '#F8BBD0')
draw_pipeline_box(13.4, 0.5, 2.3, 1.0, 'Training Data\n(~10M)', '#FFF9C4')

for x1, x2 in [(2.7, 3.5), (6.0, 6.8), (9.3, 10.1), (12.6, 13.4)]:
    ax.annotate('', xy=(x2, 1.0), xytext=(x1, 1.0),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

plt.tight_layout()
plt.show()

print("KEY POINTS to mention in the interview:")
print("  1. Bounding boxes MUST be transformed with geometric augmentations")
print("  2. Offline augmentation: faster training, more storage")
print("  3. 1M -> 10M images with augmentation")
print("  4. Split BEFORE augmentation (prevent data leakage!)")

## Phase 5: Model Architecture (15:00 - 25:00)

This is where you spend the most time. Draw the architecture on the whiteboard.

---

> **CANDIDATE**: *"Faster R-CNN has three main components. Let me draw them out.*
>
> *"First, the Backbone CNN -- typically ResNet-50. It takes the raw image (800x800x3) and produces a feature map (50x50x256). This is a compressed, learned representation of the image that captures spatial patterns like edges, textures, and shapes.*
>
> *"Second, the Region Proposal Network or RPN. This is the key innovation of Faster R-CNN. It slides a 3x3 window across the feature map. At each position, it looks at 9 anchor boxes -- 3 scales times 3 aspect ratios. For each anchor, it predicts an objectness score (is there an object here?) and bounding box adjustments (how should I refine this anchor?). That is 50 times 50 times 9 equals 22,500 anchor predictions, which get filtered down to about 300 proposals via NMS.*
>
> *"Third, the Detection Head. It takes each of the 300 proposals, crops the corresponding region from the feature map using RoI Pooling (which resizes each crop to a fixed 7x7 grid), and classifies it into one of three classes: face, license plate, or background. It also refines the bounding box coordinates.*
>
> *"For the loss function, we use a combined loss: classification cross-entropy plus lambda times the Smooth L1 regression loss for bounding boxes. Smooth L1 is preferred over MSE because it is robust to outliers -- a box that is far off does not blow up the gradient. And the regression loss is only computed on positive samples -- there is no point regressing a box for background."*

---

In [ ]:
# === Faster R-CNN Architecture Diagram (Whiteboard Style) ===

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Faster R-CNN Architecture (Whiteboard Diagram)', fontsize=15, fontweight='bold', pad=15)

def wb_box(x, y, w, h, text, color, fontsize=9):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                         facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold')

def wb_arrow(x1, y1, x2, y2, text=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))
    if text:
        mid_x = (x1 + x2) / 2
        mid_y = (y1 + y2) / 2
        ax.text(mid_x, mid_y + 0.2, text, ha='center', fontsize=8,
                style='italic', color='#555')

# Input
wb_box(0.3, 3, 2, 2, 'Input Image\n800x800x3', '#FFE0B2', 10)

# Backbone
wb_box(3.5, 3, 2.5, 2, 'Backbone\n(ResNet-50)\n\nOutput:\n50x50x256', '#C8E6C9', 9)

# RPN
wb_box(7.5, 5.5, 2.5, 2, 'RPN\n\n3x3 conv + 1x1 conv\n9 anchors/location\n22,500 proposals', '#FFCDD2', 8)

# NMS
wb_box(7.5, 3.0, 2.5, 1.5, 'NMS\n\n22,500 -> ~300\nproposals', '#FFF9C4', 9)

# RoI Pooling
wb_box(7.5, 0.5, 2.5, 1.8, 'RoI Pooling\n\n300 regions\n-> 300 x 7x7', '#B3E5FC', 9)

# Detection Head
wb_box(11.5, 1.5, 2.5, 2.5, 'Detection\nHead\n\nClassification:\nface/plate/bg\n\nBox Regression:\n(dx,dy,dw,dh)', '#D1C4E9', 8)

# Output
wb_box(11.5, 5.0, 2.5, 1.8, 'Output\n\nClass + Bbox\n+ Confidence\nfor each object', '#FFE0B2', 9)

# Arrows
wb_arrow(2.3, 4.0, 3.5, 4.0, '3 channels')
wb_arrow(6.0, 5.5, 7.5, 6.0, 'feature map')  # backbone to RPN
wb_arrow(6.0, 3.5, 7.5, 1.4, 'feature map')   # backbone to RoI pooling
wb_arrow(8.75, 5.5, 8.75, 4.5)                  # RPN to NMS
wb_arrow(8.75, 3.0, 8.75, 2.3)                  # NMS to RoI Pooling
wb_arrow(10.0, 1.4, 11.5, 2.5)                  # RoI to Detection head
wb_arrow(12.75, 4.0, 12.75, 5.0)                # Detection head to output

# Loss annotation
ax.text(14.5, 3.5, 'Loss Function:\n\nL = L_cls + lambda * L_reg\n\nL_cls = Cross-Entropy\nL_reg = Smooth L1\n(only on positives!)',
        fontsize=9, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='#FFFDE7', edgecolor='#F57F17', linewidth=2))

plt.tight_layout()
plt.show()

In [ ]:
# === Tensor Shape Walkthrough (Shows staff-level understanding) ===

print("=== Tensor Shapes Through Faster R-CNN ===")
print("(Draw this on the whiteboard to show deep understanding)\n")

shapes = [
    ("Input Image",           "(1, 3, 800, 800)",    "RGB image"),
    ("After Backbone",        "(1, 256, 50, 50)",    "Feature map (stride=16)"),
    ("RPN Objectness",        "(1, 18, 50, 50)",     "2 scores x 9 anchors"),
    ("RPN Box Deltas",        "(1, 36, 50, 50)",     "4 offsets x 9 anchors"),
    ("After NMS",             "(300, 4)",            "Top 300 proposal boxes"),
    ("After RoI Pooling",     "(300, 256, 7, 7)",    "Fixed-size feature crops"),
    ("After Flatten",         "(300, 12544)",        "256*7*7 features per proposal"),
    ("Classification Logits", "(300, 3)",            "3 classes: bg, face, plate"),
    ("Box Regression",        "(300, 12)",           "4 coords x 3 classes"),
    ("Final Detections",      "(~5-20, 6)",          "[x,y,w,h,class,conf]"),
]

print(f"{'Layer':<25} {'Shape':<22} {'Description'}")
print("-" * 75)
for layer, shape, desc in shapes:
    print(f"{layer:<25} {shape:<22} {desc}")

print("\nThis level of detail shows the interviewer you have ACTUALLY")
print("built detection models, not just read about them.")

## Phase 6: Evaluation (25:00 - 32:00)

---

> **CANDIDATE**: *"For evaluation, I would use three levels of metrics.*
>
> *"First, IoU -- Intersection over Union. This is the foundation. It measures how much a predicted box overlaps with the ground truth. An IoU above 0.5 is typically considered a correct detection.*
>
> *"Second, Average Precision or AP. We sort all predictions by confidence, go through them one by one, check if each matches a ground truth via IoU, and track precision and recall at each step. AP is the area under this precision-recall curve. This gives us a per-class quality score.*
>
> *"Third, mAP -- Mean Average Precision. We average AP across both classes: faces and license plates. This is the gold standard metric for object detection. COCO uses mAP averaged across IoU thresholds from 0.5 to 0.95.*
>
> *"For online metrics, since this is a privacy system, the primary metric is user reports -- how many people flag missed blurs. We can also use human annotators for spot-checks."*

---

In [ ]:
# === Quick Metrics Reference (Interview Crib Sheet) ===

def compute_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xi = max(x1, x2)
    yi = max(y1, y2)
    xa = min(x1+w1, x2+w2)
    ya = min(y1+h1, y2+h2)
    inter = max(0, xa-xi) * max(0, ya-yi)
    union = w1*h1 + w2*h2 - inter
    return inter/union if union > 0 else 0

# Quick IoU example
gt = [50, 50, 100, 80]
pred_good = [55, 52, 95, 78]
pred_bad = [200, 200, 50, 50]

print("=== Metrics Quick Reference ===")
print(f"\nIoU (good prediction):  {compute_iou(gt, pred_good):.4f}  -> TP at threshold 0.5")
print(f"IoU (bad prediction):   {compute_iou(gt, pred_bad):.4f}  -> FP at any threshold")
print()
print("Metric Hierarchy:")
print("  IoU       -> determines if a single prediction is correct")
print("  Precision -> fraction of correct predictions")
print("  AP        -> average precision across thresholds (per class)")
print("  mAP       -> average AP across classes (GOLD STANDARD)")
print()
print("Offline Metrics:")
print("  - mAP (overall quality)")
print("  - AP per class (face AP vs plate AP)")
print()
print("Online Metrics:")
print("  - User reports/complaints (primary)")
print("  - Human annotator spot-check rate")
print("  - (Bonus) Per-demographic recall rates for bias monitoring")

## Phase 7: Serving (32:00 - 40:00)

---

> **CANDIDATE**: *"For serving, I would design two pipelines.*
>
> *"Pipeline 1: Batch Prediction. Raw images flow through preprocessing on CPU instances -- resizing and normalizing. Then the detection model runs on GPU instances. After detection, NMS removes duplicate boxes. Finally, a blurring service applies Gaussian blur to each detected region and stores the result in object storage. The key design decision is separating CPU and GPU services so they can scale independently.*
>
> *"Pipeline 2: Data Pipeline (Feedback Loop). User reports of missed blurs get processed. The incorrectly classified images go through hard negative mining -- we take the model's confident false positives, label them as negatives, and add them to the training set. Then we retrain the model and deploy the improved version. This creates a virtuous cycle of continuous improvement.*
>
> *"For NMS specifically: we sort detections by confidence, keep the top one, remove all overlapping detections with IoU above the threshold, and repeat. It is a greedy O(N-squared) algorithm."*

---

In [ ]:
# === System Architecture Diagram (Whiteboard Version) ===

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Complete System Architecture (Whiteboard Diagram)',
             fontsize=15, fontweight='bold', pad=15)

# === BATCH PREDICTION PIPELINE ===
ax.text(8, 9.5, 'PIPELINE 1: BATCH PREDICTION', ha='center', fontsize=13,
        fontweight='bold', color='#1565C0',
        bbox=dict(boxstyle='round', facecolor='#E3F2FD'))

boxes = [
    (0.5, 7.5, 2.2, 1.3, 'Street View\nImages\n(Obj Storage)', '#FFE0B2'),
    (3.5, 7.5, 2.2, 1.3, 'Preprocess\n(CPU)\nResize, Norm', '#B3E5FC'),
    (6.5, 7.5, 2.2, 1.3, 'Detection\n(GPU)\nFaster R-CNN', '#C8E6C9'),
    (9.5, 7.5, 2.2, 1.3, 'NMS +\nBlur\n(CPU)', '#F8BBD0'),
    (12.5, 7.5, 2.2, 1.3, 'Blurred\nImages\n(Obj Storage)', '#FFE0B2'),
]

for x, y, w, h, text, color in boxes:
    wb_box(x, y, w, h, text, color, 8)

for x1, x2 in [(2.7, 3.5), (5.7, 6.5), (8.7, 9.5), (11.7, 12.5)]:
    ax.annotate('', xy=(x2, 8.15), xytext=(x1, 8.15),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Serve to users
wb_box(13.0, 6.0, 1.5, 1.0, 'Users\n(Maps)', '#FFF9C4', 9)
ax.annotate('', xy=(13.75, 7.5), xytext=(13.75, 7.0),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# === FEEDBACK PIPELINE ===
ax.text(8, 5.0, 'PIPELINE 2: FEEDBACK LOOP', ha='center', fontsize=13,
        fontweight='bold', color='#C62828',
        bbox=dict(boxstyle='round', facecolor='#FFEBEE'))

feedback_boxes = [
    (0.5, 3.0, 2.2, 1.3, 'User\nReports\n(missed blurs)', '#FFCDD2'),
    (3.5, 3.0, 2.2, 1.3, 'Review +\nLabel\nReports', '#FFF9C4'),
    (6.5, 3.0, 2.2, 1.3, 'Hard\nNegative\nMining', '#D1C4E9'),
    (9.5, 3.0, 2.2, 1.3, 'Retrain\nModel', '#C8E6C9'),
    (12.5, 3.0, 2.2, 1.3, 'Deploy\nUpdated\nModel', '#C8E6C9'),
]

for x, y, w, h, text, color in feedback_boxes:
    wb_box(x, y, w, h, text, color, 8)

for x1, x2 in [(2.7, 3.5), (5.7, 6.5), (8.7, 9.5), (11.7, 12.5)]:
    ax.annotate('', xy=(x2, 3.65), xytext=(x1, 3.65),
                arrowprops=dict(arrowstyle='->', color='#C62828', lw=2))

# Connect deployed model back to detection
ax.annotate('', xy=(7.6, 7.5), xytext=(13.6, 4.3),
            arrowprops=dict(arrowstyle='->', color='#C62828', lw=2, ls='--'))

# Connect users to reports
ax.annotate('', xy=(1.6, 4.3), xytext=(13.0, 6.0),
            arrowprops=dict(arrowstyle='->', color='#FF8F00', lw=1.5, ls='--'))
ax.text(7, 5.5, 'users report missed blurs', fontsize=8, ha='center',
        color='#FF8F00', style='italic')

# Key annotations
ax.text(4.6, 6.8, 'Scale CPU\nindependently', ha='center', fontsize=8,
        color='#0D47A1', fontweight='bold')
ax.text(7.6, 6.8, 'Scale GPU\nindependently', ha='center', fontsize=8,
        color='#1B5E20', fontweight='bold')

# Summary box
ax.text(8, 1.5, 'KEY DESIGN DECISIONS:\n'
        '1. Separate CPU/GPU services -> independent scaling\n'
        '2. Batch processing -> no latency constraint\n'
        '3. NMS after detection -> remove duplicate boxes\n'
        '4. Feedback loop -> continuous improvement via user reports\n'
        '5. Hard negative mining -> focus on worst mistakes',
        ha='center', va='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#F5F5F5', edgecolor='#333', linewidth=1.5))

plt.tight_layout()
plt.show()

## Phase 8: Follow-Up Questions (40:00 - 45:00)

This is where the interviewer tests depth. Here are the most common follow-ups and how to answer them like a staff engineer.

---

### Follow-Up 1: "Why not YOLO instead of Faster R-CNN?"

> **CANDIDATE**: *"YOLO is faster but less accurate, especially for small objects. Since we process offline with no latency constraint, speed is irrelevant -- we want maximum accuracy for privacy. However, if our dataset grows by 100x or we need real-time processing (like a live camera feed), YOLO v5 or v8 would be a strong choice. We could also use model distillation: train with Faster R-CNN, then distill into a YOLO-sized model that is nearly as accurate but much faster."*

---

### Follow-Up 2: "What about occluded or partially visible faces?"

> **CANDIDATE**: *"Partial occlusion is a real challenge. I would address it in three ways. First, during data augmentation, use random occlusion simulation -- randomly mask parts of faces during training so the model learns to detect partial faces. Second, lower the confidence threshold for production to catch more borderline cases, accepting some extra false positives. Third, leverage the feedback loop -- user reports of missed partially occluded faces become hard training examples. For very severe occlusion, the RPN should still fire if enough facial features are visible, and we can always err on the side of blurring."*

---

### Follow-Up 3: "What about GDPR and privacy regulations?"

> **CANDIDATE**: *"GDPR in Europe adds requirements beyond just blurring. Users may have the right to request deletion of their images entirely, not just blurring. We would need an opt-out mechanism where people can flag their property or themselves for complete removal. The system should also log which images were processed and when, for audit trails. Data retention policies would govern how long raw (unblurred) images are kept. Importantly, the annotated training data itself contains PII, so it must be stored securely with access controls."*

---

### Follow-Up 4: "How do you handle bias in face detection?"

> **CANDIDATE**: *"This is critical. Face detection models historically perform worse on darker skin tones, women, and children. I would address this by: (1) ensuring the training set has balanced demographic representation, (2) computing per-demographic-group recall metrics during evaluation -- not just overall mAP, (3) using hard negative/positive mining per demographic group to fix imbalances, and (4) regularly auditing the model with external fairness benchmarks. In a privacy system, any demographic gap in recall is a differential privacy violation -- certain groups would have less privacy protection than others."*

---

### Follow-Up 5: "How would you handle a 10x increase in images?"

> **CANDIDATE**: *"Three strategies. First, horizontal scaling -- add more GPU nodes since batch processing is embarrassingly parallel. Second, model optimization -- use mixed precision (FP16) for 2x speedup with minimal accuracy loss. Third, if the volume grows to billions, consider model distillation or switching to a one-stage detector like YOLOv8 that runs at 60 FPS per GPU instead of 5. We could also implement priority queues -- process high-traffic areas first and queue low-traffic rural areas."*

In [ ]:
# === Follow-Up Questions Quick Reference ===

followups = {
    "Why not YOLO?": {
        "answer": "Offline = no speed constraint. Privacy demands max accuracy. Can switch later if needed.",
        "bonus": "Mention model distillation as a compromise."
    },
    "Occluded faces?": {
        "answer": "Augmentation with random occlusion + lower confidence threshold + feedback loop.",
        "bonus": "Mention that the RPN is robust to partial features."
    },
    "GDPR compliance?": {
        "answer": "Opt-out mechanism, audit trails, data retention policies, secure training data storage.",
        "bonus": "Mention right to deletion vs right to blurring."
    },
    "Bias in face detection?": {
        "answer": "Balanced training data, per-demographic recall metrics, fairness audits.",
        "bonus": "Frame it as a differential privacy violation."
    },
    "Scale 10x?": {
        "answer": "Horizontal scaling (more GPUs), FP16, model distillation, priority queues.",
        "bonus": "Back-of-envelope calculation of GPU requirements."
    },
    "Faces in mirrors/posters/TV?": {
        "answer": "Still blur them -- they could be real people. Diverse training data helps.",
        "bonus": "When in doubt, blur. Privacy violation > false blur."
    },
    "DETR vs Faster R-CNN?": {
        "answer": "DETR eliminates anchors and NMS. Simpler but needs more data and slower to converge.",
        "bonus": "DETR treats detection as set prediction with bipartite matching."
    },
}

print("=== Common Follow-Up Questions & Answers ===")
print("(Memorize these for the interview!)\n")
for q, a in followups.items():
    print(f"Q: {q}")
    print(f"   A: {a['answer']}")
    print(f"   BONUS: {a['bonus']}")
    print()

## Scoring Rubric

Use this to evaluate your practice sessions honestly.

| Category | Excellent (5) | Good (4) | Adequate (3) | Weak (2) | Missing (1) |
|----------|--------------|----------|-------------|----------|-------------|
| **Clarification** | Asked 5+ targeted questions revealing key constraints | Asked 3-4 good questions | Asked 1-2 generic questions | Jumped to solution | No questions asked |
| **ML Framing** | Clear objective, I/O spec, justified architecture choice | Good framing, some justification | Correct but shallow | Confused detection with classification | Wrong framing |
| **Data Pipeline** | Augmentation with bbox transforms, online vs offline discussed | Mentioned augmentation and preprocessing | Basic preprocessing only | Vague hand-waving | Skipped |
| **Model Architecture** | Drew Faster R-CNN pipeline, explained RPN, loss function, tensor shapes | Explained architecture correctly | Named the components | Just said "use CNN" | No architecture discussion |
| **Evaluation** | IoU -> AP -> mAP chain, online metrics, recall vs precision trade-off | Correct metrics with some depth | Mentioned mAP | Only said "accuracy" | No metrics discussed |
| **Serving** | NMS explained, CPU/GPU separation, feedback loop, scaling | Correct pipeline with some detail | Basic pipeline | Just said "deploy model" | No serving discussion |
| **Follow-ups** | Deep, nuanced answers with trade-offs | Good answers | Surface-level answers | Struggled | Could not answer |

**Score interpretation**:
- 30-35: Strong hire
- 25-29: Hire
- 20-24: Lean hire (needs improvement)
- Below 20: Not ready yet

In [ ]:
# === Self-Assessment Tool ===

def score_interview():
    """
    Interactive scoring rubric.
    Rate yourself on each category after a practice session.
    """
    categories = [
        "Clarification (asked targeted questions)",
        "ML Framing (correct objective, I/O, architecture)",
        "Data Pipeline (augmentation, preprocessing, bbox transforms)",
        "Model Architecture (Faster R-CNN components, loss)",
        "Evaluation (IoU, AP, mAP, online metrics)",
        "Serving (NMS, pipeline, feedback loop)",
        "Follow-ups (depth, trade-offs, alternatives)",
    ]
    
    print("=== Interview Self-Assessment ===")
    print("Rate each category 1-5:\n")
    
    # Use example scores for demonstration
    example_scores = [4, 5, 4, 4, 3, 4, 3]
    total = 0
    
    for cat, score in zip(categories, example_scores):
        total += score
        bar = '#' * score + '.' * (5 - score)
        print(f"  [{bar}] {score}/5  {cat}")
    
    print(f"\n  Total: {total}/35")
    
    if total >= 30:
        verdict = "STRONG HIRE -- You are ready!"
    elif total >= 25:
        verdict = "HIRE -- Solid performance, minor areas to polish."
    elif total >= 20:
        verdict = "LEAN HIRE -- Good foundations, needs more depth."
    else:
        verdict = "NOT READY -- More practice needed. Focus on weak areas."
    
    print(f"  Verdict: {verdict}")

score_interview()

## The 30-Second Pitch

If you have 30 seconds to summarize your entire design (elevator pitch), here is what you say:

---

> *"I would design the Street View blurring system as an offline batch object detection pipeline using Faster R-CNN. The model detects faces and license plates in images, applies Non-Maximum Suppression to remove duplicate detections, and blurs the detected regions before serving. We evaluate with mAP offline and user reports online. The system separates CPU preprocessing from GPU inference for independent scaling, and uses a feedback loop with hard negative mining to continuously improve from user-reported missed blurs. The key design principle is optimizing for recall over precision -- a missed face is a privacy violation, but a false blur is harmless."*

---

That covers ALL seven steps in 30 seconds. Practice saying it out loud until it is natural.

In [ ]:
# === Key Phrases Cheat Sheet ===
# Print these on a card and review before your interview

key_phrases = [
    ("Requirements",
     '"The system prioritizes recall over precision because a missed face is a '
     'privacy violation, while a false blur is merely cosmetic."'),
    
    ("ML Framing",
     '"Object detection combines localization -- a regression task for bounding box '
     'coordinates -- with classification to identify face vs license plate."'),
    
    ("Architecture Choice",
     '"We choose Faster R-CNN because processing is offline and privacy demands maximum '
     'accuracy. If we needed real-time, YOLO would be the alternative."'),
    
    ("Data Augmentation",
     '"Critical: geometric augmentations must be applied jointly to both the image AND '
     'the bounding box annotations. Forgetting this silently ruins the model."'),
    
    ("Loss Function",
     '"L = L_cls + lambda * L_reg. Cross-entropy for classification, Smooth L1 for box '
     'regression. Regression loss is only computed on positive samples."'),
    
    ("Evaluation",
     '"mAP is the gold standard. It averages AP across classes. AP summarizes precision '
     'across IoU thresholds. User reports are the primary online metric."'),
    
    ("NMS",
     '"NMS is a greedy post-processing algorithm: keep the most confident detection, '
     'remove overlapping duplicates, repeat. O(N-squared) complexity."'),
    
    ("Serving",
     '"Separate CPU preprocessing from GPU inference for independent scaling. '
     'Batch processing since latency is not a constraint."'),
    
    ("Feedback Loop",
     '"Hard negative mining turns the model\'s confident mistakes into training examples. '
     'User reports fuel continuous improvement."'),
    
    ("Scaling",
     '"Batch processing is embarrassingly parallel. Scale horizontally by adding GPU nodes. '
     'Use FP16 for 2x speedup with minimal accuracy loss."'),
]

print("=" * 70)
print("  KEY PHRASES FOR YOUR INTERVIEW")
print("  (Drop these naturally during your answer)")
print("=" * 70)

for topic, phrase in key_phrases:
    print(f"\n[{topic}]")
    print(f"  {phrase}")

print("\n" + "=" * 70)
print("\nREMEMBER THE 7-STEP FRAMEWORK:")
print("  1. Clarify requirements")
print("  2. Frame as ML task")
print("  3. Data preparation")
print("  4. Model development")
print("  5. Evaluation (IoU -> AP -> mAP)")
print("  6. Serving (NMS + batch pipeline)")
print("  7. Discussion (DETR, GDPR, bias, scaling)")

## Summary: Complete Interview at a Glance

| Step | What to Say | Time | Key Phrase |
|------|-------------|------|------------|
| **1. Clarify** | Privacy, offline, 1M images, feedback | 4 min | "Recall over precision" |
| **2. Frame** | Object detection = regression + classification | 5 min | "Faster R-CNN for max accuracy" |
| **3. Data** | Annotated data, augmentation with bbox transforms | 5 min | "Geometric transforms must apply to bboxes too" |
| **4. Model** | Backbone + RPN + Detection Head, combined loss | 10 min | "L = L_cls + lambda * L_reg" |
| **5. Eval** | IoU -> AP -> mAP (offline), user reports (online) | 7 min | "mAP is the gold standard" |
| **6. Serve** | NMS, CPU/GPU separation, feedback loop | 8 min | "Separate CPU/GPU for independent scaling" |
| **7. Discuss** | YOLO, GDPR, bias, DETR, scaling | 5 min | "Trade-off: speed vs accuracy" |

### Final Advice

1. **Structure beats brilliance**. A well-organized average answer beats a brilliant but scattered one.
2. **Draw diagrams**. Whiteboard the architecture, the pipeline, the PR curve.
3. **State trade-offs**. Every decision has pros and cons. Show you thought about alternatives.
4. **Numbers matter**. "50x50 feature map with 9 anchors = 22,500 proposals" shows real depth.
5. **Time yourself**. The biggest failure mode is spending 20 minutes on one section and rushing the rest.
6. **Practice out loud**. Reading is not the same as speaking. Practice with a friend or a mirror.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)